In [54]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    # model="llama3.1",
    model="qwen2.5:14b",
    # temperature=0,
    # other params...
)

In [13]:
# Import relevant functionality
# from langchain_anthropic import ChatAnthropic
# from langchain_community.tools.tavily_search import TavilySearchResults
# from langchain_core.messages import HumanMessage
# from langgraph.checkpoint.sqlite import SqliteSaver
# from langgraph.prebuilt import create_react_agent


# # Create the agent
# memory = SqliteSaver.from_conn_string(":memory:")
# model = ChatAnthropic(model='ollama')
# search = TavilySearchResults(max_results=2)
# tools = [search]
# agent_executor = create_react_agent(model, tools, checkpointer=memory)

# # Use the agent
# config = {"configurable": {"thread_id": "abc123"}}
# for chunk in agent_executor.stream(
#     {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
# ):
#     print(chunk)
#     print("----")

# for chunk in agent_executor.stream(
#     {"messages": [HumanMessage(content="whats the weather where I live?")]}, config
# ):
#     print(chunk)
# #     print("----")

In [14]:
import getpass
import os

os.environ["TAVILY_API_KEY"] = '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'

In [55]:
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content="hi!")])
response.content

'Hello! How can I assist you today? Feel free to ask me anything or let me know if you have any questions.'

## 定义工具

In [56]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=2)
search_results = search.invoke("what is the weather in Chengdu")
print(search_results)
# If we want, we can create other tools.
# Once we have all the tools we want, we can put them in a list that we will reference later.
tools = [search]

[{'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Chengdu', 'region': 'Sichuan', 'country': 'China', 'lat': 30.67, 'lon': 104.07, 'tz_id': 'Asia/Shanghai', 'localtime_epoch': 1727594243, 'localtime': '2024-09-29 15:17'}, 'current': {'last_updated_epoch': 1727594100, 'last_updated': '2024-09-29 15:15', 'temp_c': 23.4, 'temp_f': 74.1, 'is_day': 1, 'condition': {'text': 'Light rain', 'icon': '//cdn.weatherapi.com/weather/64x64/day/296.png', 'code': 1183}, 'wind_mph': 18.1, 'wind_kph': 29.2, 'wind_degree': 50, 'wind_dir': 'NE', 'pressure_mb': 1007.0, 'pressure_in': 29.74, 'precip_mm': 1.04, 'precip_in': 0.04, 'humidity': 94, 'cloud': 50, 'feelslike_c': 25.6, 'feelslike_f': 78.2, 'windchill_c': 20.4, 'windchill_f': 68.7, 'heatindex_c': 20.4, 'heatindex_f': 68.7, 'dewpoint_c': 18.7, 'dewpoint_f': 65.7, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 5.0, 'gust_mph': 23.4, 'gust_kph': 37.7}}"}, {'url': 'https://tgftp.nws.noaa.gov/weather/current/ZUUU.html', 'content': 'The st

In [57]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    # model="llama3.1",
    model="qwen2.5:14b",
    # temperature=0,
    # other params...
)

In [7]:
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content="成都今天天气如何?")])
response.content

"I'm just an AI, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in Chengdu:\n\n1. Check online weather websites: You can check popular weather websites such as accuweather.com, wunderground.com, or weather.com for the current weather in Chengdu.\n2. Use a weather app: Download a weather app on your smartphone, such as Dark Sky (iOS and Android) or Weather Underground (iOS and Android), which provides real-time weather information.\n3. Check social media: You can also check social media platforms like Weibo or Sina Weibo for weather updates from local news sources or residents.\n\nAs of my knowledge cutoff in 2021, Chengdu's climate is generally mild with four distinct seasons. The city experiences a subtropical monsoon climate, characterized by hot and humid summers, cool winters, and moderate springs and autumns.\n\nIf you're looking for historical weather data or average temperature ranges for spec

In [58]:
model_with_tools = model.bind_tools(tools)

## 现在我们可以调用模型了。让我们首先用一条普通消息调用它，看看它如何响应。我们可以查看 content 字段和 tool_calls 字段。

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="Hi!")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="成都今天天气如何?")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

我们可以看到现在没有文本内容，但有一个工具调用！它希望我们调用 Tavily 搜索工具。

这还没有调用该工具 - 它只是告诉我们去调用它。为了真正调用它，我们需要创建我们的代理。

In [62]:
response = model_with_tools.invoke([HumanMessage(content="成都今天天气如何?")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': '成都今天天气'}, 'id': '50b4af00-f4da-4968-8428-1a8384b62cf6', 'type': 'tool_call'}]


# 创建代理

请注意，我们传入的是 model，而不是 model_with_tools。这是因为 create_react_agent 会在后台为我们调用 .bind_tools。

In [64]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

# 运行代理

现在我们可以对一些查询运行代理！请注意，目前，这些都是无状态查询（它不会记住之前的交互）。请注意，代理将在交互结束时返回最终状态（包括任何输入，我们将在后面看到如何只获取输出）。

首先，让我们看看当不需要调用工具时它如何响应

In [65]:
response = agent_executor.invoke({"messages": [HumanMessage(content="成都今天天气如何?")]})

response["messages"]

[HumanMessage(content='成都今天天气如何?', additional_kwargs={}, response_metadata={}, id='deaf973f-cbb5-42bd-ba42-e3bc633f8b30'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:14b', 'created_at': '2024-09-29T07:23:16.373190455Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': '成都 今天 天气'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 931965490, 'load_duration': 61407074, 'prompt_eval_count': 192, 'prompt_eval_duration': 58215000, 'eval_count': 28, 'eval_duration': 665957000}, id='run-c1afcd94-d9f6-4086-9b62-4d2a5ff01f1c-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '成都 今天 天气'}, 'id': '41d7d201-61a6-4928-9204-6c6dff6aca7c', 'type': 'tool_call'}], usage_metadata={'input_tokens': 192, 'output_tokens': 28, 'total_tokens': 220}),
 ToolMessage(content='[{"url": "https://weather.com/zh-CN/weather/today/l/464e6967d2f00205cb7f8c59cd0a592

现在让我们在一个应该调用工具的示例中尝试一下

In [63]:
response = agent_executor.invoke(
    {"messages": [HumanMessage(content="whats the weather in Chengdu?")]}
)
response["messages"]

[HumanMessage(content='whats the weather in Chengdu?', additional_kwargs={}, response_metadata={}, id='de6848e9-d1e6-4052-b17d-8962b2c41f8f'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-29T03:06:36.582814104Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': 'Chengdu weather'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 514483924, 'load_duration': 50811339, 'prompt_eval_count': 195, 'prompt_eval_duration': 86433000, 'eval_count': 24, 'eval_duration': 330660000}, id='run-6afd32a2-6638-4543-b88b-9b15030b08ed-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Chengdu weather'}, 'id': 'd1184286-fb63-4a3d-b926-c8f60d42762d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 195, 'output_tokens': 24, 'total_tokens': 219}),
 ToolMessage(content='[{"url": "https://www.weatherapi.com/", "content": "

## 流式消息

In [49]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi?")]}
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-29T03:02:46.311166714Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': 'hi?'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 501079005, 'load_duration': 42913839, 'prompt_eval_count': 190, 'prompt_eval_duration': 97982000, 'eval_count': 22, 'eval_duration': 312182000}, id='run-3f4bcacf-4366-40f4-a14d-8f189768f835-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'hi?'}, 'id': '2047a587-2b6a-497d-a10b-ba9456cbc9ab', 'type': 'tool_call'}], usage_metadata={'input_tokens': 190, 'output_tokens': 22, 'total_tokens': 212})]}}
----
{'tools': {'messages': [ToolMessage(content="HTTPError('400 Client Error: Bad Request for url: https://api.tavily.com/search')", name='tavily_search_results_json', id='928fa42e-2944-48fd-9753-cb11a2505cb0', tool

## 流式传输令牌

In [27]:
async for event in agent_executor.astream_events(
    {"messages": [HumanMessage(content="whats the weather in chengdu?")]}, version="v1"
):
    kind = event["event"]
    if kind == "on_chain_start":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print(
                f"Starting agent: {event['name']} with input: {event['data'].get('input')}"
            )
    elif kind == "on_chain_end":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print()
            print("--")
            print(
                f"Done agent: {event['name']} with output: {event['data'].get('output')['output']}"
            )
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")
    elif kind == "on_tool_start":
        print("--")
        print(
            f"Starting tool: {event['name']} with inputs: {event['data'].get('input')}"
        )
    elif kind == "on_tool_end":
        print(f"Done tool: {event['name']}")
        print(f"Tool output was: {event['data'].get('output')}")
        print("--")

--
Starting tool: tavily_search_results_json with inputs: {'query': 'weather in chengdu'}
Done tool: tavily_search_results_json
Tool output was: content='[{"url": "https://www.weatherapi.com/", "content": "{\'location\': {\'name\': \'Chengdu\', \'region\': \'Sichuan\', \'country\': \'China\', \'lat\': 30.67, \'lon\': 104.07, \'tz_id\': \'Asia/Shanghai\', \'localtime_epoch\': 1727587886, \'localtime\': \'2024-09-29 13:31\'}, \'current\': {\'last_updated_epoch\': 1727587800, \'last_updated\': \'2024-09-29 13:30\', \'temp_c\': 23.2, \'temp_f\': 73.8, \'is_day\': 1, \'condition\': {\'text\': \'Light rain\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/day/296.png\', \'code\': 1183}, \'wind_mph\': 13.0, \'wind_kph\': 20.9, \'wind_degree\': 28, \'wind_dir\': \'NNE\', \'pressure_mb\': 1007.0, \'pressure_in\': 29.74, \'precip_mm\': 1.88, \'precip_in\': 0.07, \'humidity\': 94, \'cloud\': 50, \'feelslike_c\': 25.5, \'feelslike_f\': 77.9, \'windchill_c\': 22.2, \'windchill_f\': 72.0, \'heatinde

##  添加内存

In [39]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

## It works if you put all code that depens on “memory” inside a with declaration:

In [43]:
with SqliteSaver.from_conn_string(":memory:") as memory:
  # abot = Agent(model, [tool], system=prompt, checkpointer=memory)

  # messages = [HumanMessage(content="What is the weather in sf?")]
  # thread = {"configurable": {"thread_id": "1"}}
  
  # for event in abot.graph.stream({"messages": messages}, thread):
  #     for v in event.values():
  #         print(v['messages'])
    agent_executor = create_react_agent(model, tools, checkpointer=memory)

    config = {"configurable": {"thread_id": "abc123"}}
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="hi im bob!")]}, config
    ):
        print(chunk)
        print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-29T05:54:17.299560813Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': 'definition of a current event that might be relevant for Bob'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 5480741979, 'load_duration': 4900267908, 'prompt_eval_count': 192, 'prompt_eval_duration': 99970000, 'eval_count': 31, 'eval_duration': 430248000}, id='run-e1384113-0a23-4286-8b90-a50348ab38d6-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'definition of a current event that might be relevant for Bob'}, 'id': 'cf10fbf4-8b88-4771-ab08-1a5a339ab72d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 192, 'output_tokens': 31, 'total_tokens': 223})]}}
----
{'tools': {'messages': [ToolMessage(content='[{"url": "https://library.fiveable.me/key-terms/h

In [45]:
with SqliteSaver.from_conn_string(":memory:") as memory:
  # abot = Agent(model, [tool], system=prompt, checkpointer=memory)

  # messages = [HumanMessage(content="What is the weather in sf?")]
  # thread = {"configurable": {"thread_id": "1"}}
  
  # for event in abot.graph.stream({"messages": messages}, thread):
  #     for v in event.values():
  #         print(v['messages'])
    agent_executor = create_react_agent(model, tools, checkpointer=memory)

    config = {"configurable": {"thread_id": "abc123"}}
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="hi im bob!")]}, config
    ):
        print(chunk)
        print("----")
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="whats my name?")]}, config
    ):
        print(chunk)
        print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-29T05:56:42.25030238Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': 'Bob current events'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 510936730, 'load_duration': 45020872, 'prompt_eval_count': 192, 'prompt_eval_duration': 96694000, 'eval_count': 23, 'eval_duration': 324402000}, id='run-9909807a-3a59-4680-99f8-c36bcb6da8d4-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Bob current events'}, 'id': 'e2f2e273-9f83-4ef0-a524-da22a57e8b37', 'type': 'tool_call'}], usage_metadata={'input_tokens': 192, 'output_tokens': 23, 'total_tokens': 215})]}}
----
{'tools': {'messages': [ToolMessage(content='[{"url": "https://www.npr.org/sections/news/", "content": "The Palm Beach County Court lawsuit was filed by Kim Banner, wife of Jeremy Ba

In [53]:
import json

with SqliteSaver.from_conn_string(":memory:") as memory:
  # abot = Agent(model, [tool], system=prompt, checkpointer=memory)

  # messages = [HumanMessage(content="What is the weather in sf?")]
  # thread = {"configurable": {"thread_id": "1"}}
  
  # for event in abot.graph.stream({"messages": messages}, thread):
  #     for v in event.values():
  #         print(v['messages'])
    agent_executor = create_react_agent(model, tools, checkpointer=memory)

    config = {"configurable": {"thread_id": "abc123"}}
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="hi im bob!")]}, config
    ):
        # print(chunk)
        # t = json.loads(chunk)
        # print(t)
        print("----" * 10)
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="whats my name?")]}, config
    ):
        # print(chunk)
        print("----" * 10)
    # 如果我想开始新的对话，我只需要更改使用的 thread_id
    config = {"configurable": {"thread_id": "xyz123"}}
    for chunk in agent_executor.stream(
        {"messages": [HumanMessage(content="whats my name?")]}, config
    ):
        print(chunk)
        print("----" * 10)

----------------------------------------
----------------------------------------
----------------------------------------
----------------------------------------
----------------------------------------
----------------------------------------
{'agent': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-29T06:05:21.731043385Z', 'message': {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'tavily_search_results_json', 'arguments': {'query': 'my name'}}}]}, 'done_reason': 'stop', 'done': True, 'total_duration': 474653285, 'load_duration': 60894393, 'prompt_eval_count': 192, 'prompt_eval_duration': 66318000, 'eval_count': 22, 'eval_duration': 303444000}, id='run-105bfeb9-0f5a-4f67-9d29-11c621de2315-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'my name'}, 'id': '1e4aae91-b1bf-4104-b8c4-298fb09efda9', 'type': 'tool_call'}], usage_metadata={'input_tokens': 192, 'output_

In [52]:
chunk?

Type:            AddableUpdatesDict
String form:     {'agent': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': ' <...> 'tool_call'}], usage_metadata={'input_tokens': 192, 'output_tokens': 23, 'total_tokens': 215})]}}
Length:          1
File:            ~/python-venvs/pyRobBot/lib/python3.10/site-packages/langgraph/pregel/io.py
Docstring:       <no docstring>
Class docstring: Dictionary that can be added to another dictionary.